# Clustering Comparison: Mall Customers Dataset

Comparing all 5 methods on `data/Mall_Customers.csv` (200 customers; Age,
Annual Income, Spending Score). **No ground truth labels exist** for this
dataset, so only internal (unsupervised) validity metrics apply — external
metrics (ARI/NMI/AMI/FMI) are not computed here.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_mall_customers
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "calinski_harabasz",
]
EXTERNAL_COLS = ["ari", "nmi", "ami", "fmi"]

In [ ]:
ds = load_mall_customers()
print(f"{ds.name}: X={ds.data.shape}, features={ds.feature_names}, ground_truth=None")

## Exploratory look

In [ ]:
df_raw = pd.DataFrame(ds.data, columns=ds.feature_names)
df_raw["Gender"] = ds.gender
display(df_raw.describe().T)
df_raw["Gender"].value_counts()

## 2D embedding for visualization

3 standardized features (Age, Income, Spending Score) → PCA to 2D for plotting
only; clustering runs on all 3 standardized dimensions.

In [ ]:
X_2d, xlabel, ylabel = gf.project_2d(ds.data)

## K selection (no ground truth)

Since there is no true class count to fall back on, K for K-Means++ / GMM /
Spectral Clustering is chosen by scanning k=2..10 and picking the value with
the best silhouette score.

In [ ]:
k_selection = tuning.select_kmeans(ds.data)
ks = [t["n_clusters"] for t in k_selection.tried]
best_k = k_selection.params["n_clusters"]
print("best k:", best_k)
gf.plot_k_selection(ks, [t["silhouette"] for t in k_selection.tried], best_k=best_k)
plt.show()

## DBSCAN eps selection

In [ ]:
ms = tuning.default_min_samples(ds.data.shape[1])
curve = tuning.k_distance_curve(ds.data, ms)
knee_idx, eps_at_knee = tuning.knee(curve)
gf.plot_k_distance(curve, knee_idx, eps_at_knee, min_samples=ms)
plt.show()

## Fit all 5 methods

In [ ]:
# The shared rule (tuning.select_all): K-Means k by silhouette (the number of classes when known),
# GMM by BIC, DBSCAN and HDBSCAN by the best silhouette with >= 2 clusters and <= 50% noise.
# Spectral clustering runs at the K-Means k.
selections = tuning.select_all(ds.data, k=ds.n_true_clusters, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Spectral"] = {"n_clusters": params["K-Means"]["n_clusters"], "affinity": "rbf"}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics (internal only — no ground truth)

In [ ]:
results_df = gf.build_results_table(ds.name, ds.data, results, ds.target)
results_df.to_csv(results_dir("datasets") / "mall_customers_results.csv", index=False)
display(results_df[METRIC_COLS])

## Cluster visualizations

In [ ]:
fig = gf.plot_cluster_grid(ds.data, results, y_true=None,
                           title="Mall Customers: predicted clusters (PCA 2D of Age/Income/Spending)")
plt.show()

In [ ]:
# Intuitive alternative view: Income vs. Spending Score directly (no PCA)
income_idx, spend_idx = ds.feature_names.index("Annual Income (k$)"), ds.feature_names.index("Spending Score (1-100)")
fig = gf.plot_cluster_grid(ds.data[:, [income_idx, spend_idx]], results, y_true=None,
                           title="Mall Customers: predicted clusters (Income vs. Spending Score)")
plt.show()

## Discussion

- With no ground truth, only internal metrics (Silhouette, Davies-Bouldin,
  Calinski-Harabasz) can judge cluster quality here — external metrics
  (ARI/NMI/AMI/FMI) are reported as N/A for this dataset.
- The well-known "5-6 segment" structure of this dataset (e.g. high-income/
  low-spending, low-income/high-spending, etc.) is most visually obvious in
  the raw Income-vs-Spending-Score plane, which is why that view is included
  alongside the PCA-of-all-3-features view.
- Compare how consistently each method recovers similar segment boundaries;
  density-based methods (DBSCAN/HDBSCAN) may merge or split segments
  differently than the globular-cluster assumption of K-Means++/GMM/Spectral.